In [17]:
import numpy as np

# Exercício 1:

1) Escreva um algoritmo que usa a rotina que sobrescreve uma matriz A por uma representação de sua fatoração LU sem troca de linhas (dada em aula) para resolver um sistema linear nxn.

In [18]:
def fatoracao_LU_sem_troca(matrizA: np.ndarray, b: np.ndarray):
    # Definido como arrays de floats por padrão
    n = len(matrizA)

    # Eliminação de Gauss
    for j in range(0, n - 1):  # Coluna vai até n - 1, uma a menos
        for i in range(j + 1, n):  # Linha vai até n
            fatorAlfa = (
                matrizA[i, j] / matrizA[j, j]
            )  # Calcula fator alfa dividindo pelo pivo

            matrizA[i, j] = (
                fatorAlfa  # Guarda L, fator alfa sobrescreve a posição da matriz A
            )

            for k in range(j + 1, n):  # Linha vai até n
                # O k começa em j + 1 pois NÃO PRECISAMOS calcular subtração do pivô: sabemos que vai dar zero
                matrizA[i, k] -= (
                    fatorAlfa
                    * matrizA[
                        j, k
                    ]  # Guarda U, subtração de linhas sobrescreve a posiçáo da matriz A
                )

    # Resolvendo Ly = b
    y = np.zeros(n)
    for l in range(0, n):  # Somatória de 0 a n
        soma = 0
        for m in range(0, l):
            soma += matrizA[l, m] * y[m]  # Calcula a somatória de matriz[i,j] * y[j]
        y[l] = (
            b[l] - soma
        )  # O y daquela posição será o vetor b da posição - a somatória até ali

    # Resolvendo Ux = y
    # Mesma lógica, mas começa por baixo e sobe
    x = np.zeros(n)
    for o in range(n - 1, -1, -1):
        soma = 0
        for p in range(o + 1, n):
            soma += matrizA[o, p] * x[p]
        x[o] = (y[o] - soma) / matrizA[o, o]

    return x

### Versão com Numpy (vetorizada)

Versão otimizada com vetores de Numpy, boa para vetores grandes só

In [19]:
def fatoracao_LU_sem_troca_Numpy(matrizA: np.ndarray, b: np.ndarray):
    n = len(matrizA)

    for i in range(0, n - 1):
        # Pega da linha i+1 até o fim, na coluna i (slicing)
        matrizA[i + 1 :, i] /= matrizA[
            i, i
        ]  # Divide TODOS os elementos da coluna i pelo pivô de uma só vez

        # Divisão in-place com vetor (sem precisar de loop)
        matrizA[i + 1 :, i + 1 :] -= np.outer(  # Calcula o produto externo das matrizes
            matrizA[i + 1 :, i], matrizA[i, i + 1 :]
        )  # Calcula todas as subtrações das linhas abaixo do pivô de uma vez

    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - matrizA[i, :i] @ y[:i]  # Calcula o produto escalar

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - matrizA[i, i + 1 :] @ x[i + 1 :]) / matrizA[i, i]

    return x

# Exercício 2:

2) Use seu algoritmo para resolver o sistema linear Ax=b, onde A=[1 1 0 3;2 1 -1 1;3 -1 -1 2;-1 2 3 -1] e b=[8;7;14;-7].

In [20]:
matrizA = np.array(
    [[1, 1, 0, 3], [2, 1, -1, 1], [3, -1, -1, 2], [-1, 2, 3, -1]], dtype=float
)
b = np.array([8, 7, 14, -7], dtype=float)

x = fatoracao_LU_sem_troca(matrizA, b)
print(f"Solução final x:", x)

Solução final x: [ 3. -1.  0.  2.]


# Exercício 3:

3) Adapte os códigos anteriores para desenvolver uma rotina que calcula a fatoração PA=LU e outra para resolver um sistema linear Ax=b utilizando esta fatoração. Represente a matriz de permutação P como uma permutação do vetor [1;2;...;n].

In [21]:
def fatoracao_PA_LU(matrizA: np.ndarray):
    n = len(matrizA)
    P = np.arange(1, n + 1)  # P conta de 1 a n

    for j in range(0, n - 1):
        indicePivo = (
            np.argmax(np.abs(matrizA[j:n, j])) + j
        )  # Índice do pivô vai ser o maior valor absoluto da coluna, somado a j
        if indicePivo != j:  # Se pivô não for j, vamos trocar as linhas
            matrizA[[j, indicePivo]] = matrizA[[indicePivo, j]]
            P[[j, indicePivo]] = P[[indicePivo, j]]

        for i in range(j + 1, n):
            fatorAlfa = (
                matrizA[i, j] / matrizA[j, j]
            )  # Calcula fator alfa dividindo pelo novo pivô

            matrizA[i, j] = fatorAlfa

            for k in range(j + 1, n):
                matrizA[i, k] -= fatorAlfa * matrizA[j, k]
    return P


def resolucao_linear_fatoracao(matrizA: np.ndarray, b: np.ndarray, P: np.ndarray):
    n = len(matrizA)
    b = b[
        P - 1
    ]  # Re-ordena b conforme os índices de P (subtraímos 1 pois P começa em 1)

    y = np.zeros(n)
    for l in range(0, n):
        soma = 0
        for m in range(0, l):
            soma += matrizA[l, m] * y[m]
        y[l] = b[l] - soma

    x = np.zeros(n)
    for o in range(n - 1, -1, -1):
        soma = 0
        for p in range(o + 1, n):
            soma += matrizA[o, p] * x[p]
        x[o] = (y[o] - soma) / matrizA[o, o]

    return x

### Versão com Numpy (vetorizada)

Eficiente pra matrizes grandes, continua usando vetores e produtos escalares

In [22]:
def fatoracao_PA_LU_Numpy(matrizA):
    n = len(matrizA)
    P = np.arange(1, n + 1)

    for j in range(0, n - 1):
        indicePivo = np.argmax(np.abs(matrizA[j:n, j])) + j
        if indicePivo != j:
            matrizA[[j, indicePivo]] = matrizA[[indicePivo, j]]
            P[[j, indicePivo]] = P[[indicePivo, j]]

        # Calcula todos os fatores alfa de uma vez
        matrizA[j + 1 :, j] /= matrizA[j, j]
        # Atualiza de uma vez com produto externo
        matrizA[j + 1 :, j + 1 :] -= np.outer(matrizA[j + 1 :, j], matrizA[j, j + 1 :])

    return P


def resolucao_linear_fatoracao_Numpy(matrizA: np.ndarray, b: np.ndarray, P: np.ndarray):
    n = len(matrizA)
    b = b[P - 1]

    # Substituição progressiva com produto escalar
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - matrizA[i, :i] @ y[:i]

    # Substituição regressiva com produto escalar
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - matrizA[i, i + 1 :] @ x[i + 1 :]) / matrizA[i, i]

    return x

# Exercício 4:

4) Faça passo a passo um exemplo numérico onde A=[0 1 -1 1;1 1 -1 2;-1 -1 1 0;1 2 0 2] e b=[8;-20;-2;4].

O passo-a-passo foi realizado no relatório em PDF enviado.

Verificamos o resultado obtido a partir do nosso algoritmo:

In [23]:
matrizA = np.array(
    [[0, 1, -1, 1], [1, 1, -1, 2], [-1, -1, 1, 0], [1, 2, 0, 2]], dtype=float
)
b = np.array([8, -20, -2, 4], dtype=float)

P = fatoracao_PA_LU(matrizA)
print(f"Solução final P:", P)

x = resolucao_linear_fatoracao(matrizA, b, P)
print(f"Solução final x:", x)

Solução final P: [2 1 4 3]
Solução final x: [-17.   21.5   2.5 -11. ]
